In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool 
from langchain.agents import create_agent 

In [27]:
loader = PyPDFLoader("../Data/0121YG008021121_190858c.pdf")
docs = loader.load()

In [28]:
len(docs)

11

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

In [30]:
len(splitted_docs)

41

In [32]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [33]:
same_record = vector_store.similarity_search("Patient name")

In [9]:
same_record[0].page_content

'DIAGNOSTIC REPORT\nPLOT NO.1, LANE NO. 1,  SITUATED AT HAZURI\nBAGH ROAD, TALAB TILLO, \nPATIENT NAME :  AYUSH BHAT\nPATIENT ID       : AYUSM217552130\nACCESSION NO : 0121YG008021 AGE/SEX    :22 Years Male\nABHA NO          :\nDRAWN      :24/07/2025 12:21:00\nRECEIVED  :24/07/2025 13:02:03\nREPORTED  :24/07/2025 17:25:30\nREF. DOCTOR :  SELF\nCLIENT PATIENT ID:\nCODE/NAME & ADDRESS  : C000079918\nSAI MULTISPECIALITY\nJAMMU 180002\n7006048100 9419191754\nFinal Results Biological Reference IntervalTest Report Status Units\nBIOCHEMISTRY\nCOMPLETE CARE SCREEN\nGLUCOSE FASTING,FLUORIDE PLASMA\nFBS (FASTING BLOOD SUGAR) 81 (Normal <100,Impaired fasting\nglucose:100 to 125,Diabetes\nmellitus:>=126(on more than\n1 occasion)(ADA guidelines\n2024)\nmg/dL\nKIDNEY FUNCTION TEST, SERUM\nBLOOD UREA NITROGEN 10 6 - 20 mg/dL\nCREATININE 0.69  Low 0.90 - 1.30 mg/dL\nBUN/CREAT RATIO 14.49 10 - 20\nURIC ACID 4.9 3.7 - 9.2 mg/dL\nTOTAL PROTEIN 7.7 5.7 - 8.2 g/dL\nALBUMIN 4.8 3.2 - 4.8 g/dL\nGLOBULIN 2.9 

#### AGENT = tools , llm , prompt 

In [ ]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve relevant data of PDF Documents associated with your medical tests mentioning all the details ...
    """
    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query,k=4)
    context = ""
    
    for doc in docs : 
        context = doc.page_content + "\n\n"
        
    return context
    # print(len(docs), docs[0].page_content)
    

In [44]:
# retriever_tool.invoke("Patient Name")
llm = ChatOpenAI(model="gpt-5")

In [46]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""


In [47]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [48]:
query = "What is the name of patient and doctor"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

Tool Called: patient name and doctor name


In [50]:
result = response["messages"][-1].content

In [51]:
print(result)

- Patient name: Ayush Bhat
- Doctor name: Self (no specific doctor listed)
